# Week 4 — Fine-tune TrOCR + evaluate with CER/WER

**Before running any code this week: Runtime > Change runtime type > GPU.**

This notebook drives `src/train.py` and `src/evaluate.py`. See `src/train.py`'s
docstring for the full list of changes vs. the handout's Cells 1-3 (Seq2SeqTrainer
with `predict_with_generate`, cosine LR + warmup, more epochs with early
stopping on CER, mixed precision, beam-search decoding). The short version:
these changes exist because they measurably reduce Character Error Rate, not
just to make the code longer.


In [ ]:
# Run once per Colab session (safe to re-run: pulls latest instead of
# silently reusing a stale checkout from an earlier run this session).
import os
import subprocess
import sys

REPO_DIR = '/content/repo'
if not os.path.exists(REPO_DIR):
    # Replace with your own fork's URL if you're running this outside the
    # original solution repo.
    !git clone https://github.com/Rameesha-Malik/ML-SI-Solution-Urdu-OCR.git {REPO_DIR}
else:
    print(f'{REPO_DIR} already exists -- pulling latest instead of re-cloning.')
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)

!pip install -q --upgrade -r requirements.txt
# Force transformers (and only transformers -- not the whole requirements
# file, so this doesn't also re-download the much larger torch wheel) to
# the pinned exact version. A plain `-r requirements.txt` install can
# leave an already-installed version in place if pip's resolver treats it
# as already 'satisfying' things.
!pip install -q --upgrade --force-reinstall "transformers==4.46.3"

# Ground-truth check via subprocess (`pip show`), NOT `import transformers`
# -- if a cell anywhere earlier in this session already imported
# transformers, Python caches that in memory for the rest of the process,
# and reinstalling on disk doesn't change what's already loaded here. The
# actual training/eval scripts run as a separate `!python ...` subprocess
# each time, so they always see what's genuinely on disk -- this mirrors
# that instead of this notebook kernel's own (possibly stale) view.
result = subprocess.run(['pip', 'show', 'transformers'], capture_output=True, text=True)
installed_version = next(
    (l.split(':', 1)[1].strip() for l in result.stdout.splitlines() if l.startswith('Version:')),
    '?',
)
print('transformers installed on disk:', installed_version)

if 'src' not in sys.path:
    sys.path.insert(0, 'src')

if not installed_version.startswith('4.'):
    raise RuntimeError(
        f'transformers on disk is {installed_version}, not the pinned 4.46.3 -- '
        'pip could not satisfy the pin here, likely another package in this '
        'Colab image requires transformers>=5. Re-run this cell and share the '
        'full pip output above so the real conflicting package can be found.'
    )
print('Setup complete -- transformers', installed_version, 'is correctly installed on disk.')


In [ ]:
import torch
print('Using device:', 'cuda' if torch.cuda.is_available() else 'cpu')
if not torch.cuda.is_available():
    print('WARNING: No GPU detected. Go to Runtime > Change runtime type > GPU.')


## Train

Adjust `--epochs` / `--batch-size` for your Colab tier. Training logs CER/WER after every epoch (unlike the handout's loop, which only prints training loss).

In [ ]:
!python src/train.py \
    --train-csv data/train.csv --test-csv data/test.csv \
    --output-dir /content/drive/MyDrive/SI26-urdu-ocr-model \
    --epochs 12 --batch-size 8 --grad-accum 2


(If you'd rather keep checkpoints local to the Colab VM instead of Drive, drop the `/content/drive/...` path and use `models/urdu-trocr` — just remember to save to Drive/HF Hub before the session ends.)

## Evaluate

Reports Character Error Rate, Word Error Rate, and exact-match accuracy (the handout's original metric) side by side, plus the 5 worst predictions for your write-up.

In [ ]:
!python src/evaluate.py /content/drive/MyDrive/SI26-urdu-ocr-model/final \
    --test-csv data/test.csv \
    --report-csv reports/eval_report.csv


**What to record** (same ask as the handout, now with real numbers instead of just exact-match): your CER / WER / exact-match accuracy, and 3-5 wrong examples from `reports/eval_report.csv` (sort by the `cer` column). These go into your Week 5 README's Results section.

## Save your model

Already saved to Drive by `--output-dir` above during training. To also publish it to the HuggingFace Hub (recommended — Spaces deployment in Week 5 can then load it with `MODEL_PATH=yourusername/your-model-name`, no Drive mount needed):

In [ ]:
from huggingface_hub import notebook_login
notebook_login()  # paste a HuggingFace token with write access

from transformers import TrOCRProcessor, VisionEncoderDecoderModel

model_dir = '/content/drive/MyDrive/SI26-urdu-ocr-model/final'
repo_id = 'yourusername/urdu-trocr-si26'  # <-- change this

model = VisionEncoderDecoderModel.from_pretrained(model_dir)
processor = TrOCRProcessor.from_pretrained(model_dir)
model.push_to_hub(repo_id)
processor.push_to_hub(repo_id)
print(f'Model published at: https://huggingface.co/{repo_id}')
